# Fabric Managed Private Endpoint Manager

Inventory and (optionally) delete managed private endpoints across one or
more Fabric workspaces, using the official `managedPrivateEndpoints` REST API.

## Workflow

1. **Configure** (cell 1) — pick scope, filters, and safety knobs.
2. **Inventory** (cell 3) — discover every MPE you can see and persist it to a Delta table + JSON/CSV under `Files/`.
3. **Dry run** (cell 4) — show exactly which endpoints would be deleted.
4. **Commit** (cell 5) — delete them only when `COMMIT=True` AND the matched count is under `MAX_DELETES`.
5. **Recreate** (cell 6) — rebuild MPEs from the saved inventory/audit when `RECREATE=True`. New MPEs come up in `Provisioning` / `Pending` state.
6. **Approve** (cell 7) — auto-approve the resulting Pending PECs on the Azure side when `APPROVE=True`. Uses an ARM token (different audience and RBAC than Fabric); matches PECs by a run marker stamped into the request message, so only PECs from THIS run are touched.

## Safety

- `COMMIT=False` by default → cell 5 is a no-op.
- `MAX_DELETES` aborts cell 5 if your filter matches more than the cap.
- Every delete is appended to an audit Delta table with the API status code + response body.
- The inventory is always persisted before any delete attempt.

## Prerequisites

- A default Lakehouse is attached to this notebook.
- The session identity has **Admin** role on every workspace you want to delete from (Viewer is enough for inventory).
- For cell 7 (approve) only: the identity must also have `Microsoft.<Provider>/<resourceType>/privateEndpointConnections/write` (typically `Owner` or `Contributor`) on each **target Azure resource** — Fabric Workspace Admin is not enough.

## Authentication

Three options, tried in order by `get_token()` (for Fabric REST):

1. `MPE_TOKEN` environment variable — useful for unattended runs and for the SPN sample in cell 2b.
2. `notebookutils.credentials.getToken('pbi')` — default in Fabric.
3. `az account get-access-token` — when running locally with `az login`.

**Cell 7 needs a different token** (ARM audience `https://management.azure.com`). `get_arm_token()` tries (in order):

1. `ARM_TOKEN` environment variable.
2. `notebookutils.credentials.getToken("https://management.azure.com/")` and a few audience variants — **usually fails in Fabric** because the workspace identity isn't allowed to mint ARM tokens; included for completeness.
3. `ARM_SPN_TENANT_ID` + `ARM_SPN_CLIENT_ID` + `ARM_SPN_CLIENT_SECRET` env vars — client_credentials grant directly against ARM. **This is the reliable path in Fabric.** Cell 2b sets these for you when its `SPN_*` vars are filled in.
4. `az account get-access-token --resource https://management.azure.com` — local only.

If none work, the error lists exactly what was tried so you can pick a path.

For scheduled / pipeline runs, fill in the three `SPN_*` variables in **cell 2b** to authenticate as a service principal. Cell 2b will populate both the Fabric token (for cells 3-6) and the ARM token (for cell 7) from the same SPN, provided that SPN has RBAC on both sides.


In [ ]:
"""Configuration — edit these knobs, then run the cells in order."""
from datetime import datetime, timezone

# ---- Workspace scope ------------------------------------------------------
# "visible"        -> every workspace the caller can see
# "list"           -> just the IDs in WORKSPACES below
# "from_inventory" -> reload from the latest matching run in the Delta table
WORKSPACE_SCOPE = "visible"
WORKSPACES      = []        # used when WORKSPACE_SCOPE = "list"

# ---- Persistence ----------------------------------------------------------
INVENTORY_TABLE = "mpe_inventory"        # Delta table for inventories
AUDIT_TABLE     = "mpe_delete_audit"     # Delta table for delete results
WRITE_SCHEMA    = ""                     # optional Spark schema/database name
FILES_SUBDIR    = "mpe_manager"          # where JSON/CSV land under Files/

# ---- Filters (applied in cells 4 and 5) ----------------------------------
NAME_FILTER     = None                   # regex string over MPE display name
ID_FILTER       = []                     # explicit MPE IDs to include
TARGET_FILTER   = None                   # regex over targetPrivateLinkResourceId

# ---- Deletion safety -----------------------------------------------------
COMMIT          = False                  # set True ONLY when you really want to delete
MAX_DELETES     = 25                     # aborts cell 5 if matches exceed this

# ---- Recreate (cell 6) ---------------------------------------------------
RECREATE                 = False         # set True to actually call POST /managedPrivateEndpoints
RECREATE_SOURCE          = "audit"       # "audit" (recreate what we deleted) | "inventory"
RECREATE_RUN_LABEL       = None          # None -> use current RUN_LABEL
RECREATE_REQUEST_MESSAGE = "Recreated via mpe_manager.ipynb"
RECREATE_TABLE           = "mpe_recreate_audit"
MAX_RECREATES            = 25

# ---- Approve (cell 7) ----------------------------------------------------
APPROVE                  = False         # set True to PUT Approved on Azure-side PECs
APPROVE_DESCRIPTION      = "Approved by mpe_manager"
APPROVE_RUN_LABEL        = None          # None -> use current RUN_LABEL
APPROVE_TABLE            = "mpe_approve_audit"
MAX_APPROVES             = 25

# ---- API / Auth ----------------------------------------------------------
FABRIC_BASE     = "https://api.fabric.microsoft.com"
ARM_BASE        = "https://management.azure.com"  # cell 7 (PEC approval)
TOKEN_AUDIENCE  = "pbi"                  # notebookutils.credentials.getToken(...)

# ---- Auto-generated ------------------------------------------------------
RUN_LABEL       = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S")

print(f"Run label   : {RUN_LABEL}")
print(f"Scope       : {WORKSPACE_SCOPE}")
print(f"Commit mode : {COMMIT}   (max_deletes={MAX_DELETES})")
print(f"Recreate    : {RECREATE} (source={RECREATE_SOURCE!r}, "
      f"max_recreates={MAX_RECREATES})")
print(f"Inventory   : {WRITE_SCHEMA + '.' if WRITE_SCHEMA else ''}{INVENTORY_TABLE}")
print(f"Audit       : {WRITE_SCHEMA + '.' if WRITE_SCHEMA else ''}{AUDIT_TABLE}")
print(f"Recreate audit: {WRITE_SCHEMA + '.' if WRITE_SCHEMA else ''}{RECREATE_TABLE}")
print(f"Approve     : {APPROVE} (max_approves={MAX_APPROVES})")
print(f"Approve audit: {WRITE_SCHEMA + '.' if WRITE_SCHEMA else ''}{APPROVE_TABLE}")


In [ ]:
"""Core API helpers — token, paged list, delete, filter.

Kept inline so this notebook is self-contained. Mirrors the surface of the
standalone `fabric_mpe_manager.py` CLI minus the argparse/main wrapper.
"""
import json as _json
import os as _os
import re as _re
import subprocess as _subprocess
import time as _time
import urllib.error as _urlerr
import urllib.parse as _urlparse
import urllib.request as _urlreq
from datetime import datetime, timezone


def get_token():
    tok = _os.environ.get("MPE_TOKEN")
    if tok:
        return tok.strip()
    try:
        import notebookutils  # type: ignore
        return notebookutils.credentials.getToken(TOKEN_AUDIENCE)
    except Exception:
        pass
    try:
        out = _subprocess.check_output(
            ["az", "account", "get-access-token",
             "--resource", FABRIC_BASE,
             "--query", "accessToken", "-o", "tsv"],
            stderr=_subprocess.PIPE, text=True,
        ).strip()
        if out:
            return out
    except Exception:
        pass
    raise RuntimeError(
        "Could not acquire a Fabric token. In Fabric this should come from "
        "notebookutils.credentials.getToken('pbi'); locally, run `az login`.")


def _req(method, url, token, body=None, max_retries=5):
    data = _json.dumps(body).encode() if body is not None else None
    headers = {"Authorization": f"Bearer {token}",
               "Accept": "application/json"}
    if body is not None:
        headers["Content-Type"] = "application/json"
    for attempt in range(max_retries):
        req = _urlreq.Request(url, data=data, method=method, headers=headers)
        try:
            with _urlreq.urlopen(req, timeout=60) as r:
                raw = r.read().decode("utf-8")
                return r.status, (_json.loads(raw) if raw.strip() else {})
        except _urlerr.HTTPError as e:
            if e.code == 429:
                wait = int(e.headers.get("Retry-After", "5"))
                print(f"  ! 429 rate-limited, sleeping {wait}s")
                _time.sleep(wait)
                continue
            if 500 <= e.code < 600 and attempt + 1 < max_retries:
                _time.sleep(min(2 ** attempt, 30))
                continue
            err_body = e.read().decode("utf-8", errors="replace")
            try:
                err_parsed = _json.loads(err_body)
            except Exception:
                err_parsed = {"raw": err_body}
            return e.code, err_parsed
        except _urlerr.URLError:
            if attempt + 1 < max_retries:
                _time.sleep(2 ** attempt)
                continue
            raise
    return 0, {"error": "exhausted retries"}


def list_workspaces(token):
    url = f"{FABRIC_BASE}/v1/workspaces"
    out = []
    while url:
        status, data = _req("GET", url, token)
        if status != 200:
            raise RuntimeError(
                f"List workspaces failed: HTTP {status} {data}")
        out.extend(data.get("value", []))
        ct = data.get("continuationToken")
        url = (f"{FABRIC_BASE}/v1/workspaces?"
               f"continuationToken={_urlparse.quote(ct)}") if ct else None
    return out


def list_mpes(workspace_id, token):
    base = (f"{FABRIC_BASE}/v1/workspaces/{workspace_id}"
            f"/managedPrivateEndpoints")
    url = base
    out = []
    while url:
        status, data = _req("GET", url, token)
        if status == 200:
            out.extend(data.get("value", []))
            ct = data.get("continuationToken")
            url = (f"{base}?continuationToken={_urlparse.quote(ct)}"
                   if ct else None)
            continue
        return [], {"status": status, "body": data}
    return out, None


def delete_mpe(workspace_id, mpe_id, token):
    url = (f"{FABRIC_BASE}/v1/workspaces/{workspace_id}"
           f"/managedPrivateEndpoints/{mpe_id}")
    return _req("DELETE", url, token)


def create_mpe(workspace_id, body, token):
    """POST /v1/workspaces/{wid}/managedPrivateEndpoints.

    body must include `name` and `targetPrivateLinkResourceId`; optional
    keys: `targetSubresourceType`, `requestMessage`, `targetFQDNs`.
    Returns (status, parsed_response).
    """
    url = (f"{FABRIC_BASE}/v1/workspaces/{workspace_id}"
           f"/managedPrivateEndpoints")
    return _req("POST", url, token, body=body)


def apply_filters(inventory, *, name_filter=None,
                  id_filter=None, target_filter=None):
    out = inventory
    if id_filter:
        wanted = set(id_filter)
        out = [m for m in out if m.get("mpe_id") in wanted]
    if name_filter:
        rx = _re.compile(name_filter)
        out = [m for m in out if rx.search(m.get("mpe_name") or "")]
    if target_filter:
        rx = _re.compile(target_filter)
        out = [m for m in out if rx.search(m.get("target_resource_id") or "")]
    return out


def _delta_target(table):
    return f"{WRITE_SCHEMA}.{table}" if WRITE_SCHEMA else table


def _files_path(subdir):
    return f"/lakehouse/default/Files/{subdir}"


# ---- Azure ARM helpers (cell 7 — PEC approval) ---------------------------
# api-version dispatch for {targetResourceId}/privateEndpointConnections.
# Conservative GA versions; override the dict if your RP isn't listed.
_PEC_API_VERSIONS = {
    "Microsoft.Storage/storageAccounts":              "2023-05-01",
    "Microsoft.Sql/servers":                          "2023-08-01-preview",
    "Microsoft.KeyVault/vaults":                      "2023-07-01",
    "Microsoft.DocumentDB/databaseAccounts":          "2023-04-15",
    "Microsoft.EventHub/namespaces":                  "2024-01-01",
    "Microsoft.ServiceBus/namespaces":                "2022-10-01-preview",
    "Microsoft.Synapse/workspaces":                   "2021-06-01",
    "Microsoft.Search/searchServices":                "2023-11-01",
    "Microsoft.CognitiveServices/accounts":           "2023-05-01",
    "Microsoft.AppConfiguration/configurationStores": "2023-03-01",
    "Microsoft.Web/sites":                            "2023-12-01",
    "Microsoft.ContainerRegistry/registries":         "2023-11-01-preview",
    "Microsoft.DataFactory/factories":                "2018-06-01",
    "Microsoft.Purview/accounts":                     "2021-12-01",
}
_PEC_DEFAULT_API_VERSION = "2023-09-01"


def _rp_from_resource_id(rid):
    """/.../providers/<RP>/<type>/<name>[/...]  ->  'Microsoft.X/typeY'."""
    if not rid:
        return None
    parts = rid.split("/providers/", 1)
    if len(parts) < 2:
        return None
    seg = parts[1].split("/")
    if len(seg) < 2:
        return None
    return f"{seg[0]}/{seg[1]}"


def _pec_api_version(rid):
    rp = _rp_from_resource_id(rid)
    return _PEC_API_VERSIONS.get(rp, _PEC_DEFAULT_API_VERSION), rp


def get_arm_token():
    """Acquire an ARM token (audience https://management.azure.com).

    In Fabric, the workspace identity often CANNOT mint an ARM token via
    notebookutils — the documented audiences are pbi/storage/keyvault. So
    this function tries (in order):

      1. ARM_TOKEN env var
      2. notebookutils with several audience variants (works in some runtimes)
      3. SPN client_credentials via ARM_SPN_TENANT_ID / ARM_SPN_CLIENT_ID /
         ARM_SPN_CLIENT_SECRET env vars (the reliable path in Fabric)
      4. az CLI (works locally)

    Every failure is captured and included in the final error message so the
    user knows exactly what was tried.
    """
    tok = _os.environ.get("ARM_TOKEN")
    if tok:
        return tok.strip()

    errors = []
    arm_scope = ARM_BASE.rstrip("/")

    try:
        import notebookutils  # type: ignore
    except Exception as e:
        notebookutils = None
        errors.append(f"import notebookutils: {type(e).__name__}: {e}")

    if notebookutils is not None:
        for aud in (arm_scope, arm_scope + "/",
                    "azuremanagement", "management"):
            try:
                t = notebookutils.credentials.getToken(aud)
                if t:
                    return t
                errors.append(f"notebookutils.getToken({aud!r}): empty")
            except Exception as e:
                errors.append(
                    f"notebookutils.getToken({aud!r}): "
                    f"{type(e).__name__}: {e}")

    spn_tid = _os.environ.get("ARM_SPN_TENANT_ID")
    spn_cid = _os.environ.get("ARM_SPN_CLIENT_ID")
    spn_sec = _os.environ.get("ARM_SPN_CLIENT_SECRET")
    if spn_tid and spn_cid and spn_sec:
        try:
            return _fetch_token_via_spn(
                spn_tid, spn_cid, spn_sec,
                scope=f"{arm_scope}/.default")
        except Exception as e:
            errors.append(
                f"SPN client_credentials (ARM_SPN_*): "
                f"{type(e).__name__}: {e}")

    try:
        out = _subprocess.check_output(
            ["az", "account", "get-access-token",
             "--resource", arm_scope,
             "--query", "accessToken", "-o", "tsv"],
            stderr=_subprocess.PIPE, text=True,
        ).strip()
        if out:
            return out
        errors.append("az CLI: empty token returned")
    except Exception as e:
        errors.append(f"az CLI: {type(e).__name__}: {e}")

    detail = "\n  - " + "\n  - ".join(errors) if errors else ""
    raise RuntimeError(
        "Could not acquire an ARM token.\n\n"
        "In Fabric the workspace identity typically cannot mint ARM tokens "
        "via notebookutils. To run cell 7 (Azure-side approval), pick one:\n"
        "  (a) Paste a pre-acquired ARM token into the ARM_TOKEN env var "
        "(e.g. from `az account get-access-token "
        "--resource https://management.azure.com`), OR\n"
        "  (b) Set ARM_SPN_TENANT_ID / ARM_SPN_CLIENT_ID / "
        "ARM_SPN_CLIENT_SECRET env vars (cell 2b sets these for you when "
        "SPN_* are filled in), OR\n"
        "  (c) Run the notebook locally with `az login`.\n\n"
        "Attempts tried:" + detail)


def _fetch_token_via_spn(tenant_id, client_id, client_secret, scope):
    """Generic client_credentials grant. Used by get_arm_token and cell 2b."""
    url = (f"https://login.microsoftonline.com/{tenant_id}"
           "/oauth2/v2.0/token")
    body = _urlparse.urlencode({
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": scope,
    }).encode("utf-8")
    req = _urlreq.Request(url, data=body, method="POST")
    req.add_header("Content-Type", "application/x-www-form-urlencoded")
    with _urlreq.urlopen(req, timeout=30) as resp:
        payload = _json.loads(resp.read())
    if "access_token" not in payload:
        raise RuntimeError(
            f"token endpoint did not return access_token: {payload}")
    return payload["access_token"]


def list_pecs(target_resource_id, token):
    """LIST privateEndpointConnections on a target Azure resource.

    Returns (status, payload_or_list, rp, api_version). On 200 the second
    element is a list of PEC objects; otherwise it's the error body.
    """
    api, rp = _pec_api_version(target_resource_id)
    url = (f"{ARM_BASE}{target_resource_id}/privateEndpointConnections"
           f"?api-version={api}")
    out = []
    while url:
        status, data = _req("GET", url, token)
        if status != 200:
            return status, data, rp, api
        out.extend(data.get("value", []))
        url = data.get("nextLink")
    return 200, out, rp, api


def approve_pec(target_resource_id, pec_name, token,
                description="Approved", api_version=None):
    """PUT Approved on a single privateEndpointConnection.

    api_version is auto-derived from target_resource_id if not given.
    """
    if api_version is None:
        api_version, _ = _pec_api_version(target_resource_id)
    url = (f"{ARM_BASE}{target_resource_id}/privateEndpointConnections/"
           f"{pec_name}?api-version={api_version}")
    body = {"properties": {"privateLinkServiceConnectionState": {
        "status": "Approved",
        "description": description,
    }}}
    return _req("PUT", url, token, body=body)


# Fail fast on broken auth before the user runs the inventory cell.
TOKEN = get_token()
print("Token acquired.")


In [ ]:
"""OPTIONAL: Service Principal (app registration) authentication sample.

By default the notebook calls `notebookutils.credentials.getToken('pbi')` in
Fabric, or `az account get-access-token` locally. For unattended/scheduled
runs use an Azure AD application instead.

PREREQUISITES (one-time setup):
  1. Tenant admin enables "Service principals can use Fabric APIs"
     (Admin portal -> Tenant settings -> Developer settings).
  2. Add the SPN as a **Workspace Admin** on every workspace you want to
     scan (Viewer is enough for inventory-only, Admin is required for
     delete + create).
  3. Store the client secret in a Key Vault and grant the SPN
     (or the workspace identity) read access.

To use this cell:
  * Fill in SPN_TENANT_ID, SPN_CLIENT_ID, and SPN_CLIENT_SECRET below
    (or read the secret from Key Vault — see the commented block).
  * Run this cell BEFORE cell 3 (inventory). The token is written to the
    MPE_TOKEN environment variable, which the existing get_token() picks up
    first, so the rest of the notebook needs no other changes.

Leave the three SPN_* variables empty to skip this cell entirely (no-op).
"""
import json as _json_spn
import os as _os_spn
import urllib.parse as _urlp_spn
import urllib.request as _urlr_spn


def fabric_token_via_spn(tenant_id, client_id, client_secret,
                         scope="https://api.fabric.microsoft.com/.default"):
    """Client-credentials grant against Entra; returns a Fabric bearer token.

    `scope` defaults to the Fabric REST audience. Swap to
    "https://analysis.windows.net/powerbi/api/.default" if your tenant still
    requires the legacy Power BI audience.
    """
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    body = _urlp_spn.urlencode({
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": scope,
    }).encode("utf-8")
    req = _urlr_spn.Request(url, data=body, method="POST")
    req.add_header("Content-Type", "application/x-www-form-urlencoded")
    with _urlr_spn.urlopen(req, timeout=30) as resp:
        payload = _json_spn.loads(resp.read())
    if "access_token" not in payload:
        raise RuntimeError(f"Token endpoint did not return access_token: {payload}")
    return payload["access_token"]


# ---- Fill these in to enable SPN auth -------------------------------------
SPN_TENANT_ID     = ""    # e.g. "11111111-2222-3333-4444-555555555555"
SPN_CLIENT_ID     = ""    # SPN (app registration) client id (also a GUID)
SPN_CLIENT_SECRET = ""    # paste only for ad-hoc tests; production = pull from KV

# ---- Production pattern: read secret from a Key Vault (recommended) -------
# Uncomment ONE of these blocks. Both use the workspace's managed identity
# (no secret in the notebook), so you only need to grant that identity
# "Get" on the KV secret.
#
#   # Fabric / Synapse — workspace identity reads from KV
#   import notebookutils
#   SPN_CLIENT_SECRET = notebookutils.credentials.getSecret(
#       "https://<your-kv>.vault.azure.net",
#       "<secret-name>")
#
#   # Or read from an env var injected by the pipeline / orchestrator
#   import os
#   SPN_CLIENT_SECRET = os.environ.get("SPN_CLIENT_SECRET", "")
# ---------------------------------------------------------------------------

if SPN_TENANT_ID and SPN_CLIENT_ID and SPN_CLIENT_SECRET:
    _os_spn.environ["MPE_TOKEN"] = fabric_token_via_spn(
        SPN_TENANT_ID, SPN_CLIENT_ID, SPN_CLIENT_SECRET)
    # Re-run the module's acquisition so cells 3+ use the SPN token even
    # though they captured TOKEN at import time.
    TOKEN = get_token()
    print("SPN Fabric token: acquired and exported as MPE_TOKEN; "
          "TOKEN refreshed.")
    # Also surface the same SPN to get_arm_token() (cell 7).
    _os_spn.environ["ARM_SPN_TENANT_ID"]     = SPN_TENANT_ID
    _os_spn.environ["ARM_SPN_CLIENT_ID"]     = SPN_CLIENT_ID
    _os_spn.environ["ARM_SPN_CLIENT_SECRET"] = SPN_CLIENT_SECRET
    # And pre-fetch the ARM token so cell 7 doesn't have to. Non-fatal if
    # the SPN lacks ARM RBAC — cell 7 will surface that error itself.
    try:
        _os_spn.environ["ARM_TOKEN"] = fabric_token_via_spn(
            SPN_TENANT_ID, SPN_CLIENT_ID, SPN_CLIENT_SECRET,
            scope="https://management.azure.com/.default")
        print("SPN ARM token:    acquired and exported as ARM_TOKEN "
              "(cell 7 will use it).")
    except Exception as _spn_arm_exc:
        print(f"SPN ARM token:    skipped ({type(_spn_arm_exc).__name__}: "
              f"{_spn_arm_exc}).\n"
              "  Cell 7 will retry via SPN env vars (ARM_SPN_*) when run.")
else:
    print("SPN sample skipped — SPN_TENANT_ID / SPN_CLIENT_ID / "
          "SPN_CLIENT_SECRET not set. Notebook will keep using the default "
          "notebookutils / az CLI token from cell 2.")


In [ ]:
"""Cell 3 — inventory MPEs and persist to Delta + Files (JSON+CSV)."""
import csv as _csv
import json as _json
import os as _os

# Resolve which workspaces to scan.
workspace_names = {}
if WORKSPACE_SCOPE == "visible":
    ws = list_workspaces(TOKEN)
    workspace_ids = [w["id"] for w in ws if w.get("id")]
    workspace_names = {w["id"]: w.get("displayName") for w in ws}
    print(f"Discovered {len(workspace_ids)} visible workspace(s).")
elif WORKSPACE_SCOPE == "list":
    if not WORKSPACES:
        raise RuntimeError("WORKSPACE_SCOPE='list' but WORKSPACES is empty.")
    workspace_ids = list(WORKSPACES)
    print(f"Using {len(workspace_ids)} workspace ID(s) from config.")
elif WORKSPACE_SCOPE == "from_inventory":
    prior = spark.read.table(_delta_target(INVENTORY_TABLE))
    workspace_ids = sorted({r.workspace_id for r in
                            prior.select("workspace_id").distinct().collect()})
    print(f"Replaying {len(workspace_ids)} workspace(s) from prior inventory.")
else:
    raise RuntimeError(f"Unknown WORKSPACE_SCOPE: {WORKSPACE_SCOPE!r}")

inventory_rows = []
skipped = []
for i, wid in enumerate(workspace_ids, 1):
    eps, err = list_mpes(wid, TOKEN)
    if err is not None:
        skipped.append((wid, err))
        print(f"  [{i}/{len(workspace_ids)}] {wid}: skip "
              f"({err.get('status')})")
        continue
    for ep in eps:
        cs = ep.get("connectionState") or {}
        inventory_rows.append({
            "workspace_id": wid,
            "workspace_name": workspace_names.get(wid),
            "mpe_id": ep.get("id"),
            "mpe_name": ep.get("name"),
            "target_resource_id": ep.get("targetPrivateLinkResourceId"),
            "target_subresource_type": ep.get("targetSubresourceType"),
            "provisioning_state": ep.get("provisioningState"),
            "connection_status": cs.get("status"),
            "connection_description": cs.get("description"),
            "run_label": RUN_LABEL,
            "collected_at": datetime.now(timezone.utc),
        })
    if eps:
        print(f"  [{i}/{len(workspace_ids)}] {wid}: {len(eps)} MPE(s)")

print(f"\nInventory complete: {len(inventory_rows)} MPE(s); "
      f"{len(skipped)} workspace(s) skipped.")

# Persist JSON + CSV under Files/ for export.
files_dir = _files_path(f"{FILES_SUBDIR}/{RUN_LABEL}")
_os.makedirs(files_dir, exist_ok=True)
inv_json = _os.path.join(files_dir, "inventory.json")
inv_csv = _os.path.join(files_dir, "inventory.csv")

_inv_for_files = [{**r, "collected_at": r["collected_at"].isoformat()}
                  for r in inventory_rows]
with open(inv_json, "w", encoding="utf-8") as f:
    _json.dump(_inv_for_files, f, indent=2)
csv_fields = ["workspace_id", "workspace_name", "mpe_id", "mpe_name",
              "target_resource_id", "target_subresource_type",
              "provisioning_state", "connection_status",
              "connection_description", "run_label", "collected_at"]
with open(inv_csv, "w", newline="", encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=csv_fields)
    w.writeheader()
    for row in _inv_for_files:
        w.writerow({k: row.get(k) for k in csv_fields})
print(f"Wrote {inv_json}")
print(f"Wrote {inv_csv}")

# Append to Delta and surface as displays.
if inventory_rows:
    inv_df = spark.createDataFrame(inventory_rows)
    (inv_df.write.format("delta").mode("append")
        .saveAsTable(_delta_target(INVENTORY_TABLE)))
    print(f"Appended {len(inventory_rows)} row(s) to "
          f"{_delta_target(INVENTORY_TABLE)}")
    inv_df.createOrReplaceTempView("inv_v")

    print("\n=== This run's inventory ===")
    display(spark.sql(
        "SELECT workspace_name, workspace_id, mpe_name, mpe_id, "
        "       target_subresource_type, provisioning_state, "
        "       connection_status, target_resource_id "
        "FROM inv_v ORDER BY workspace_name, mpe_name"))

    print("\n=== Rollup by target subresource type ===")
    display(spark.sql(
        "SELECT target_subresource_type, COUNT(*) AS n "
        "FROM inv_v GROUP BY target_subresource_type ORDER BY n DESC"))

    print("\n=== Rollup by provisioning / connection state ===")
    display(spark.sql(
        "SELECT provisioning_state, connection_status, COUNT(*) AS n "
        "FROM inv_v "
        "GROUP BY provisioning_state, connection_status "
        "ORDER BY n DESC"))
else:
    print("(No MPEs found for this scope.)")


In [ ]:
"""Cell 4 — DRY RUN. Shows exactly which MPEs cell 5 would delete.

Does NOT call delete_mpe. Safe to run at any time.
"""
inv = (spark.read.table(_delta_target(INVENTORY_TABLE))
       .where(f"run_label = '{RUN_LABEL}'"))
inventory_rows = [r.asDict() for r in inv.collect()]
print(f"Loaded {len(inventory_rows)} MPE(s) from this run "
      f"({_delta_target(INVENTORY_TABLE)}).")

targets = apply_filters(
    inventory_rows,
    name_filter=NAME_FILTER,
    id_filter=ID_FILTER or None,
    target_filter=TARGET_FILTER,
)

print(f"\nFilters match {len(targets)} MPE(s):")
for m in targets[:50]:
    print(f"  - ws={m['workspace_id']} mpe={m['mpe_id']} "
          f"({m.get('mpe_name')!r}) -> {m.get('target_resource_id')}")
if len(targets) > 50:
    print(f"  ... and {len(targets) - 50} more")

if not targets:
    print("\n(Nothing matches; cell 5 will be a no-op.)")
elif len(targets) > MAX_DELETES:
    print(f"\n!! Match count ({len(targets)}) exceeds MAX_DELETES "
          f"({MAX_DELETES}). Cell 5 will REFUSE to run until you narrow "
          "the filter or raise the cap in cell 1.")
else:
    print(f"\nReady. Set COMMIT=True in cell 1, re-run cell 1, then run "
          f"cell 5 to delete these {len(targets)} MPE(s).")


In [ ]:
"""Cell 5 — COMMIT. Deletes MPEs and logs every result to the audit table.

Gated by `COMMIT=True` in cell 1. Aborts if matches exceed MAX_DELETES.
"""
inv = (spark.read.table(_delta_target(INVENTORY_TABLE))
       .where(f"run_label = '{RUN_LABEL}'"))
inventory_rows = [r.asDict() for r in inv.collect()]
targets = apply_filters(
    inventory_rows,
    name_filter=NAME_FILTER,
    id_filter=ID_FILTER or None,
    target_filter=TARGET_FILTER,
)

if not COMMIT:
    print(f"COMMIT=False; skipping. {len(targets)} MPE(s) would be deleted.")
elif not targets:
    print("Nothing matches the filter; nothing to delete.")
elif len(targets) > MAX_DELETES:
    raise RuntimeError(
        f"ABORT: {len(targets)} matches exceed MAX_DELETES "
        f"({MAX_DELETES}). Narrow the filter or raise the cap before "
        "re-running.")
else:
    audit_rows = []
    ok = err = 0
    for i, m in enumerate(targets, 1):
        status, body = delete_mpe(m["workspace_id"], m["mpe_id"], TOKEN)
        audit_rows.append({
            "workspace_id": m["workspace_id"],
            "workspace_name": m.get("workspace_name"),
            "mpe_id": m["mpe_id"],
            "mpe_name": m.get("mpe_name"),
            "target_resource_id": m.get("target_resource_id"),
            "target_subresource_type": m.get("target_subresource_type"),
            "provisioning_state": m.get("provisioning_state"),
            "connection_status": m.get("connection_status"),
            "delete_status": int(status),
            "delete_response": (_json.dumps(body)
                                if not isinstance(body, str) else body),
            "run_label": RUN_LABEL,
            "deleted_at": datetime.now(timezone.utc),
        })
        if status in (200, 202, 204):
            ok += 1
            print(f"  [{i}/{len(targets)}] {m['mpe_id']}: deleted ({status})")
        else:
            err += 1
            print(f"  [{i}/{len(targets)}] {m['mpe_id']}: "
                  f"FAILED ({status}) {body}")

    audit_df = spark.createDataFrame(audit_rows)
    (audit_df.write.format("delta").mode("append")
        .saveAsTable(_delta_target(AUDIT_TABLE)))
    print(f"\nDone: {ok} deleted, {err} failed. Audit appended to "
          f"{_delta_target(AUDIT_TABLE)}.")
    audit_df.createOrReplaceTempView("audit_v")
    print("\n=== This run's audit ===")
    display(spark.sql(
        "SELECT workspace_name, mpe_name, mpe_id, delete_status, "
        "       target_resource_id, deleted_at "
        "FROM audit_v ORDER BY delete_status, mpe_name"))


In [ ]:
"""Cell 6 — RECREATE. Rebuilds MPEs from a saved inventory or audit row.

Default source is the audit table from this run (i.e., the MPEs cell 5 just
deleted). Set RECREATE_SOURCE='inventory' to rebuild from any inventory
snapshot — useful for restoring across workspaces or after the fact.

Gated by `RECREATE=True` in cell 1. Aborts if matches exceed MAX_RECREATES.
The new MPEs come up in 'Provisioning' / 'Pending' state and must be
re-approved on the Azure resource side (Private Link Center).
"""
target_run_label = RECREATE_RUN_LABEL or RUN_LABEL

if RECREATE_SOURCE not in ("audit", "inventory"):
    raise RuntimeError(
        f"Unknown RECREATE_SOURCE: {RECREATE_SOURCE!r} "
        "(expected 'audit' or 'inventory')")

src_table = _delta_target(
    AUDIT_TABLE if RECREATE_SOURCE == "audit" else INVENTORY_TABLE)

# Defensive: if the source table doesn't exist yet (e.g. nothing has ever
# been deleted, so the audit table was never created), give a friendly
# message instead of a TABLE_OR_VIEW_NOT_FOUND stack trace.
if not spark.catalog.tableExists(src_table):
    hint = ("Run cell 5 with COMMIT=True first to populate the audit "
            "table, or set RECREATE_SOURCE='inventory' (in cell 1) "
            "to rebuild from a prior inventory snapshot."
            if RECREATE_SOURCE == "audit"
            else "Run cell 3 first to populate the inventory table.")
    msg = f"Source table '{src_table}' does not exist yet. {hint}"
    if RECREATE:
        raise RuntimeError(msg)
    print(f"{msg}\n(RECREATE=False — preview only. Skipping.)")
    src_rows = []
else:
    src = spark.read.table(src_table).where(
        f"run_label = '{target_run_label}'")
    if RECREATE_SOURCE == "audit":
        src = src.where("delete_status BETWEEN 200 AND 299")
    src_rows = [r.asDict() for r in src.collect()]

print(f"Source {RECREATE_SOURCE!r} (run_label={target_run_label}): "
      f"{len(src_rows)} candidate row(s)")

# Apply name + target filters; skip ID_FILTER because IDs change on recreate.
targets = apply_filters(
    src_rows,
    name_filter=NAME_FILTER,
    target_filter=TARGET_FILTER,
)
print(f"Filters match {len(targets)} MPE(s).")

if not RECREATE:
    print(f"\nRECREATE=False; skipping. Would create {len(targets)} MPE(s):")
    for r in targets[:50]:
        print(f"  - ws={r['workspace_id']} name={r.get('mpe_name')!r} "
              f"-> {r.get('target_resource_id')}")
    if len(targets) > 50:
        print(f"  ... and {len(targets) - 50} more")
elif not targets:
    print("Nothing matches the filter; nothing to recreate.")
elif len(targets) > MAX_RECREATES:
    raise RuntimeError(
        f"ABORT: {len(targets)} matches exceed MAX_RECREATES "
        f"({MAX_RECREATES}). Narrow the filter or raise the cap before "
        "re-running.")
else:
    recreate_rows = []
    ok = err = 0
    # Stamp every requestMessage with a run marker so cell 7 can match the
    # resulting Pending PECs back to this exact run.
    run_marker = f"[run={RUN_LABEL}]"
    for i, r in enumerate(targets, 1):
        body = {
            "name": r.get("mpe_name"),
            "targetPrivateLinkResourceId": r.get("target_resource_id"),
        }
        if r.get("target_subresource_type"):
            body["targetSubresourceType"] = r["target_subresource_type"]
        base_msg = RECREATE_REQUEST_MESSAGE or "Recreated via mpe_manager.ipynb"
        body["requestMessage"] = f"{run_marker} {base_msg}"[:140]

        status, resp = create_mpe(r["workspace_id"], body, TOKEN)
        new_id = (resp.get("id")
                  if isinstance(resp, dict) and resp.get("id") else None)
        new_state = (resp.get("provisioningState")
                     if isinstance(resp, dict) else None)
        recreate_rows.append({
            "workspace_id": r["workspace_id"],
            "workspace_name": r.get("workspace_name"),
            "original_mpe_id": r.get("mpe_id"),
            "new_mpe_id": new_id,
            "mpe_name": r.get("mpe_name"),
            "target_resource_id": r.get("target_resource_id"),
            "target_subresource_type": r.get("target_subresource_type"),
            "request_message": RECREATE_REQUEST_MESSAGE,
            "create_status": int(status),
            "create_response": (_json.dumps(resp)
                                if not isinstance(resp, str) else resp),
            "new_provisioning_state": new_state,
            "source": RECREATE_SOURCE,
            "source_run_label": target_run_label,
            "run_label": RUN_LABEL,
            "created_at": datetime.now(timezone.utc),
        })
        if status in (200, 201, 202):
            ok += 1
            print(f"  [{i}/{len(targets)}] {r.get('mpe_name')!r}: "
                  f"created (new_id={new_id}, state={new_state})")
        else:
            err += 1
            print(f"  [{i}/{len(targets)}] {r.get('mpe_name')!r}: "
                  f"FAILED ({status}) {resp}")

    recreate_df = spark.createDataFrame(recreate_rows)
    (recreate_df.write.format("delta").mode("append")
        .saveAsTable(_delta_target(RECREATE_TABLE)))
    print(f"\nDone: {ok} created, {err} failed. "
          f"Audit appended to {_delta_target(RECREATE_TABLE)}.")
    recreate_df.createOrReplaceTempView("recreate_v")
    print("\n=== This run's recreate audit ===")
    display(spark.sql(
        "SELECT workspace_name, mpe_name, original_mpe_id, new_mpe_id, "
        "       create_status, new_provisioning_state, "
        "       target_resource_id, created_at "
        "FROM recreate_v ORDER BY create_status, mpe_name"))
    print("\nNOTE: New MPEs start in 'Pending' connection state. Run cell 7 "
          "to auto-approve them on the Azure side, or approve manually in "
          "Private Link Center > Pending connections.")


In [ ]:
"""Cell 7 — APPROVE. Auto-approves the Azure-side privateEndpointConnections.

After cell 6 recreates MPEs they come up Pending on the target Azure resource.
This cell:
  1. Reads `mpe_recreate_audit` for the chosen run_label.
  2. Groups by target_resource_id and LISTs PECs on each via ARM REST.
  3. Filters to status == 'Pending' AND description contains the run marker
     that cell 6 stamped in ('[run=<RUN_LABEL>]'). This guarantees we only
     touch PECs created by THIS notebook in THIS run.
  4. If APPROVE=True (and the queue is <= MAX_APPROVES), PUTs Approved on
     each one, writes the result to `mpe_approve_audit`, and displays the
     audit. Otherwise prints a preview and exits.

AUTH: Uses a separate ARM token (audience https://management.azure.com),
not the Fabric token. The caller needs Owner / a role with
`Microsoft.<RP>/<resourceType>/privateEndpointConnections/write` on EACH
target Azure resource — Fabric Workspace Admin is not enough.

CROSS-TENANT: If the target resource lives in a different tenant than the
Fabric workspace, you'll need an ARM token from that other tenant — set
the ARM_TOKEN env var to a token for that tenant before running this cell.
"""
target_run_label = APPROVE_RUN_LABEL or RUN_LABEL
recreate_tbl = _delta_target(RECREATE_TABLE)
marker_substr = f"[run={target_run_label}]"

if not spark.catalog.tableExists(recreate_tbl):
    msg = (f"RECREATE_TABLE '{recreate_tbl}' does not exist yet. "
           "Run cell 6 with RECREATE=True first.")
    if APPROVE:
        raise RuntimeError(msg)
    print(f"{msg}\n(APPROVE=False — preview only. Skipping.)")
    candidates = []
else:
    rec = (spark.read.table(recreate_tbl)
           .where(f"run_label = '{target_run_label}'")
           .where("create_status BETWEEN 200 AND 299"))
    candidates = [r.asDict() for r in rec.collect()]

print(f"Recreate audit (run_label={target_run_label}): "
      f"{len(candidates)} successful recreate(s).")

by_target = {}
for r in candidates:
    by_target.setdefault(r["target_resource_id"], []).append(r)

queue = []
list_errors = []
arm_token = get_arm_token() if candidates else None

for tgt, rows in by_target.items():
    status, payload, rp, api = list_pecs(tgt, arm_token)
    if status != 200:
        list_errors.append({"target_resource_id": tgt, "rp": rp,
                            "api_version": api, "status": status,
                            "payload": payload})
        print(f"  ! LIST failed on {tgt}: HTTP {status} {payload}")
        continue
    pending = [
        p for p in payload
        if (p.get("properties", {})
              .get("privateLinkServiceConnectionState", {})
              .get("status") == "Pending")
        and marker_substr in (
            p.get("properties", {})
              .get("privateLinkServiceConnectionState", {})
              .get("description") or "")
    ]
    print(f"  - {tgt}: {len(pending)} pending PEC(s) match marker "
          f"'{marker_substr}' (rp={rp}, api={api})")
    for p in pending:
        queue.append({
            "target_resource_id": tgt,
            "rp": rp,
            "api_version": api,
            "pec_name": p["name"],
            "pec_description": (p.get("properties", {})
                                  .get("privateLinkServiceConnectionState", {})
                                  .get("description")),
            "mpe_rows": rows,
        })

print(f"\nTotal pending PECs queued for approval: {len(queue)}")
if list_errors:
    print(f"LIST errors: {len(list_errors)} (see prints above).")

if not APPROVE:
    print("\nAPPROVE=False — preview only. Skipping.")
    for q in queue[:50]:
        print(f"  Would approve: {q['target_resource_id']}/{q['pec_name']}")
    if len(queue) > 50:
        print(f"  ... and {len(queue) - 50} more")
elif not queue and not list_errors:
    print("Nothing to approve.")
elif len(queue) > MAX_APPROVES:
    raise RuntimeError(
        f"ABORT: {len(queue)} queued exceed MAX_APPROVES "
        f"({MAX_APPROVES}). Narrow the run_label or raise the cap.")
else:
    approve_rows = []
    ok = err = 0
    for i, q in enumerate(queue, 1):
        linked = q["mpe_rows"][0] if q["mpe_rows"] else {}
        status, resp = approve_pec(
            q["target_resource_id"], q["pec_name"], arm_token,
            description=APPROVE_DESCRIPTION, api_version=q["api_version"])
        new_state = (resp.get("properties", {})
                       .get("privateLinkServiceConnectionState", {})
                       .get("status")
                     if isinstance(resp, dict) else None)
        approve_rows.append({
            "target_resource_id": q["target_resource_id"],
            "target_rp": q["rp"],
            "api_version": q["api_version"],
            "pec_name": q["pec_name"],
            "pec_description": q["pec_description"],
            "mpe_name": linked.get("mpe_name"),
            "original_mpe_id": linked.get("original_mpe_id"),
            "new_mpe_id": linked.get("new_mpe_id"),
            "request_message": linked.get("request_message"),
            "approve_status": int(status),
            "approve_response": (_json.dumps(resp)
                                 if not isinstance(resp, str) else resp),
            "new_connection_state": new_state,
            "source_run_label": target_run_label,
            "run_label": RUN_LABEL,
            "approved_at": datetime.now(timezone.utc),
        })
        if status in (200, 201, 202):
            ok += 1
            print(f"  [{i}/{len(queue)}] {q['pec_name']}: "
                  f"approved (state={new_state})")
        else:
            err += 1
            print(f"  [{i}/{len(queue)}] {q['pec_name']}: "
                  f"FAILED ({status}) {resp}")

    for le in list_errors:
        approve_rows.append({
            "target_resource_id": le["target_resource_id"],
            "target_rp": le["rp"],
            "api_version": le["api_version"],
            "pec_name": None,
            "pec_description": None,
            "mpe_name": None,
            "original_mpe_id": None,
            "new_mpe_id": None,
            "request_message": None,
            "approve_status": int(le["status"]),
            "approve_response": (_json.dumps(le["payload"])
                                 if not isinstance(le["payload"], str)
                                 else le["payload"]),
            "new_connection_state": None,
            "source_run_label": target_run_label,
            "run_label": RUN_LABEL,
            "approved_at": datetime.now(timezone.utc),
        })

    if approve_rows:
        approve_df = spark.createDataFrame(approve_rows)
        (approve_df.write.format("delta").mode("append")
            .saveAsTable(_delta_target(APPROVE_TABLE)))
        print(f"\nDone: {ok} approved, {err} failed, "
              f"{len(list_errors)} LIST error(s). "
              f"Audit appended to {_delta_target(APPROVE_TABLE)}.")
        approve_df.createOrReplaceTempView("approve_v")
        print("\n=== This run's approve audit ===")
        display(spark.sql(
            "SELECT pec_name, mpe_name, new_mpe_id, approve_status, "
            "       new_connection_state, target_rp, target_resource_id, "
            "       approved_at "
            "FROM approve_v ORDER BY approve_status, pec_name"))


## How to use

1. **Edit cell 1** to set scope, filters, and safety knobs. Leave `COMMIT=False` and `RECREATE=False` for now.
1a. **(Optional) Cell 2b** — for unattended runs, fill in the three `SPN_*` variables to authenticate as a service principal. Leave them empty and the notebook uses notebookutils (in Fabric) or `az login` (locally).
2. **Run cells 1 → 2 → 2b → 3** to inventory MPEs. The inventory is written to:
   - `Files/<FILES_SUBDIR>/<RUN_LABEL>/inventory.json`
   - `Files/<FILES_SUBDIR>/<RUN_LABEL>/inventory.csv`
   - Delta table `<INVENTORY_TABLE>` (append, with `run_label`)
3. **Run cell 4** to dry-run with your filters. Nothing is deleted.
4. When the dry-run looks right, **edit cell 1**: set `COMMIT=True`, re-run cell 1 to reload, then **run cell 5**. Failure of the cap check raises an error.
5. To **recreate** what you just deleted: set `RECREATE=True` in cell 1, re-run cell 1, then **run cell 6**. Defaults to recreating every MPE the audit table says you successfully deleted in this run.
6. To **auto-approve** the resulting Pending PECs on the Azure side: set `APPROVE=True` in cell 1, re-run cell 1, then **run cell 7**. Approve uses a separate ARM token (audience `https://management.azure.com`) and looks up each PEC by the run marker (`[run=<RUN_LABEL>]`) that cell 6 stamps into the request message.

## Recreate scenarios

| Goal | `RECREATE_SOURCE` | `RECREATE_RUN_LABEL` |
|---|---|---|
| Undo what cell 5 just deleted | `"audit"` (default) | `None` (defaults to current `RUN_LABEL`) |
| Re-stand-up MPEs from an older delete run | `"audit"` | the old `run_label` string |
| Replay an inventory snapshot (e.g., reseed a new workspace) | `"inventory"` | a saved inventory's `run_label` |

The recreate cell posts `{name, targetPrivateLinkResourceId, targetSubresourceType, requestMessage}` to the create API. New MPEs come up in `Provisioning` state with a `Pending` connection — the Azure resource owner must approve each one under **Private Link Center > Pending connections**.

## Tables

### `<INVENTORY_TABLE>` (default `mpe_inventory`)

| column | type | notes |
|---|---|---|
| `workspace_id` / `workspace_name` | string | |
| `mpe_id` / `mpe_name` | string | |
| `target_resource_id` | string | `targetPrivateLinkResourceId` |
| `target_subresource_type` | string | `blob`, `sqlServer`, `dfs`, etc. |
| `provisioning_state` | string | `Succeeded`, `Provisioning`, `Failed`, ... |
| `connection_status` | string | `Approved`, `Pending`, `Rejected`, ... |
| `connection_description` | string | |
| `run_label` | string | identifies the inventory batch |
| `collected_at` | timestamp | UTC |

### `<AUDIT_TABLE>` (default `mpe_delete_audit`)

Inventory columns plus:

| column | type | notes |
|---|---|---|
| `delete_status` | int | HTTP status from the DELETE call |
| `delete_response` | string | JSON body (or raw text) returned |
| `deleted_at` | timestamp | UTC |
| `run_label` | string | matches the inventory's run_label |

### `<RECREATE_TABLE>` (default `mpe_recreate_audit`)

| column | type | notes |
|---|---|---|
| `workspace_id` / `workspace_name` | string | |
| `original_mpe_id` | string | the pre-delete MPE id (may not exist anymore) |
| `new_mpe_id` | string | id returned by the create call (NULL on failure) |
| `mpe_name` / `target_resource_id` / `target_subresource_type` | string | what was sent to create |
| `request_message` | string | prefixed with `[run=<RUN_LABEL>]` so cell 7 can match it |
| `create_status` | int | HTTP status from the POST call |
| `create_response` | string | full JSON body |
| `new_provisioning_state` | string | typically `Provisioning` immediately after create |
| `source` | string | `audit` or `inventory` |
| `source_run_label` | string | run_label of the row we recreated from |
| `run_label` | string | this run's label |
| `created_at` | timestamp | UTC |

### `<APPROVE_TABLE>` (default `mpe_approve_audit`)

| column | type | notes |
|---|---|---|
| `target_resource_id` | string | the Azure resource the PEC sits on |
| `target_rp` | string | e.g. `Microsoft.Storage/storageAccounts` |
| `api_version` | string | api-version used for the PUT (or LIST on error) |
| `pec_name` | string | the Azure-side connection name (auto-generated) |
| `pec_description` | string | the connection's description = the requestMessage we set in cell 6 |
| `mpe_name` / `original_mpe_id` / `new_mpe_id` | string | linkage back to the Fabric MPE |
| `request_message` | string | what cell 6 originally posted |
| `approve_status` | int | HTTP status from the PUT call (LIST errors also recorded with their status) |
| `approve_response` | string | full JSON body |
| `new_connection_state` | string | `Approved` on success |
| `source_run_label` | string | run_label of the recreate run we approved for |
| `run_label` | string | this run's label |
| `approved_at` | timestamp | UTC |

## Useful follow-up queries

```sql
-- All MPEs you've ever inventoried, with the most recent record per (workspace, mpe)
SELECT * FROM mpe_inventory
WHERE (workspace_id, mpe_id, collected_at) IN (
  SELECT workspace_id, mpe_id, MAX(collected_at)
  FROM mpe_inventory GROUP BY workspace_id, mpe_id);

-- MPEs deleted in the last 7 days
SELECT * FROM mpe_delete_audit
WHERE deleted_at > current_timestamp() - INTERVAL 7 DAYS
  AND delete_status BETWEEN 200 AND 299;

-- Recreate failures that need attention
SELECT * FROM mpe_recreate_audit
WHERE create_status >= 400
ORDER BY created_at DESC;

-- Round-trip view: deleted -> recreated -> approved
SELECT d.mpe_name, d.workspace_name,
       d.mpe_id AS old_id, r.new_mpe_id,
       d.deleted_at, r.created_at, a.approved_at,
       r.create_status, a.approve_status, a.new_connection_state
FROM mpe_delete_audit d
LEFT JOIN mpe_recreate_audit r ON r.original_mpe_id = d.mpe_id
LEFT JOIN mpe_approve_audit  a ON a.new_mpe_id      = r.new_mpe_id
WHERE d.delete_status BETWEEN 200 AND 299
ORDER BY d.deleted_at DESC;
```

## API reference

| Operation | Endpoint | Role |
|---|---|---|
| List MPEs | `GET /v1/workspaces/{wid}/managedPrivateEndpoints` | Viewer |
| Create MPE | `POST /v1/workspaces/{wid}/managedPrivateEndpoints` | **Admin** (Fabric) |
| Delete MPE | `DELETE /v1/workspaces/{wid}/managedPrivateEndpoints/{mpeId}` | **Admin** (Fabric) |
| List PECs (Azure) | `GET {targetResourceId}/privateEndpointConnections?api-version=...` | Reader on target resource |
| Approve PEC (Azure) | `PUT {targetResourceId}/privateEndpointConnections/{name}?api-version=...` body `{properties.privateLinkServiceConnectionState.status: "Approved"}` | `*/privateEndpointConnections/write` on target resource |

All endpoints honor 429 `Retry-After` and 5xx exponential backoff via the shared `_req` helper.
